In [ ]:
!ls "/content/physionet.org/files/mimic-iv-note/2.2/note"

discharge.csv	      index.html     radiology_detail.csv
discharge_detail.csv  radiology.csv


In [ ]:
!ls "/content/mimiciv_50pct_bgot_truncated/physionet.org_extracted/files/mimiciv/3.1/hosp"
!ls "/content/mimiciv_50pct_bgot_truncated/physionet.org_extracted/files/mimiciv/3.1/icu"

admissions.csv	   microbiologyevents.csv  poe.csv	       transfers.csv
diagnoses_icd.csv  omr.csv		   poe_detail.csv
emar.csv	   patients.csv		   prescriptions.csv
labevents.csv	   pharmacy.csv		   procedures_icd.csv
chartevents.csv  inputevents.csv   procedureevents.csv
icustays.csv	 outputevents.csv


In [ ]:
import os
import pandas as pd

print("="*60)
print("DATASET STATISTICS")
print("="*60)

# ======== SET PATHS (from your unzip output) ========
STRUCT_PATH = "/content/mimiciv_50pct_bgot_truncated/physionet.org_extracted/files/mimiciv/3.1"
NOTE_PATH = "/content/physionet.org/files/mimic-iv-note/2.2/note"

hosp = STRUCT_PATH + "/hosp"
icu = STRUCT_PATH + "/icu"

# =========================
# 1. STRUCTURED DATA
# =========================
print("\n--- STRUCTURED (MIMIC-IV) ---")

patients = pd.read_csv(f"{hosp}/patients.csv")
admissions = pd.read_csv(f"{hosp}/admissions.csv")
icustays = pd.read_csv(f"{icu}/icustays.csv")

print(f"Patients: {len(patients):,}")
print(f"Admissions: {len(admissions):,}")
print(f"ICU stays: {len(icustays):,}")

# =========================
# 2. NOTES DATA
# =========================
print("\n--- NOTES (MIMIC-IV-NOTE) ---")

discharge_path = f"{NOTE_PATH}/discharge.csv"
radiology_path = f"{NOTE_PATH}/radiology.csv"

if os.path.exists(discharge_path):
    discharge = pd.read_csv(discharge_path)
    print(f"Discharge notes: {len(discharge):,}")
    print(f"Unique hadm_ids (discharge): {discharge['hadm_id'].nunique():,}")
else:
    print("Discharge notes: NOT FOUND")

if os.path.exists(radiology_path):
    radiology = pd.read_csv(radiology_path)
    print(f"Radiology reports: {len(radiology):,}")
    print(f"Unique hadm_ids (radiology): {radiology['hadm_id'].nunique():,}")
else:
    print("Radiology reports: NOT FOUND")

# =========================
# 3. QUICK SIZE CHECK
# =========================
print("\n--- FILE SIZES ---")

def size_gb(path):
    if os.path.exists(path):
        return os.path.getsize(path) / 1e9
    return 0.0

print(f"labevents.csv: {size_gb(f'{hosp}/labevents.csv'):.2f} GB")
print(f"chartevents.csv: {size_gb(f'{icu}/chartevents.csv'):.2f} GB")
print(f"discharge.csv: {size_gb(discharge_path):.2f} GB")

DATASET STATISTICS

--- STRUCTURED (MIMIC-IV) ---
Patients: 36,411
Admissions: 43,050
ICU stays: 45,232

--- NOTES (MIMIC-IV-NOTE) ---
Discharge notes: 331,793
Unique hadm_ids (discharge): 331,793
Radiology reports: 2,321,355
Unique hadm_ids (radiology): 309,670

--- FILE SIZES ---
labevents.csv: 2.62 GB
chartevents.csv: 21.91 GB
discharge.csv: 3.53 GB


In [ ]:
# =========================
# Install dependency
# =========================
!pip install -q huggingface_hub

# =========================
# Imports
# =========================
from huggingface_hub import snapshot_download
import os
import json

# =========================
# CONFIG
# =========================
REPO_ID = "GOVINDFROM/temporal-epistemic-graph-of-thought"

# =========================
# DOWNLOAD REPO
# =========================
local_path = snapshot_download(repo_id=REPO_ID)
print(f"\nDownloaded to: {local_path}\n")

# =========================
# PRINT DIRECTORY TREE
# =========================
def print_tree(root, prefix=""):
    files = sorted(os.listdir(root))
    for i, name in enumerate(files):
        path = os.path.join(root, name)
        connector = "└── " if i == len(files) - 1 else "├── "
        print(prefix + connector + name)
        if os.path.isdir(path):
            extension = "    " if i == len(files) - 1 else "│   "
            print_tree(path, prefix + extension)

print("===== FULL DIRECTORY STRUCTURE =====\n")
print_tree(local_path)

# =========================
# LOCATE v15 FOLDER
# =========================
v15_path = os.path.join(local_path, "tegot_outputs_v15")

print(f"\n===== v15 PATH =====\n{v15_path}\n")

# =========================
# LIST FILES INSIDE v15
# =========================
print("===== FILES INSIDE tegot_outputs_v15 =====\n")
for root, dirs, files in os.walk(v15_path):
    level = root.replace(v15_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

# =========================
# LOAD RESULTS FILES
# =========================
results_path = os.path.join(v15_path, "results")

def load_json(fname):
    fpath = os.path.join(results_path, fname)
    if os.path.exists(fpath):
        with open(fpath, "r") as f:
            data = json.load(f)
        print(f"\n===== {fname} =====")
        print(json.dumps(data, indent=2)[:3000])  # truncated view
    else:
        print(f"\n{fname} NOT FOUND")

print("\n===== LOADING RESULTS =====")

load_json("test_metrics.json")
load_json("val_metrics.json")
load_json("baselines.json")
load_json("ablations.json")
load_json("training_histories.json")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 75 files:   0%|          | 0/75 [00:00<?, ?it/s]


Downloaded to: /root/.cache/huggingface/hub/models--GOVINDFROM--temporal-epistemic-graph-of-thought/snapshots/bda267103ba7a0530668d99a3d9de41cbe51e5b5

===== FULL DIRECTORY STRUCTURE =====

├── .gitattributes
├── checkpoints
│   ├── tegot_s1337.pt
│   └── tegot_s42.pt
├── cohort.csv
├── cohort_sub.csv
├── models
│   └── tegot_best.pt
├── plots
│   ├── fig1_training.png
│   ├── fig2_uncertainty.png
│   ├── fig3_structural.png
│   ├── fig4_selective.png
│   ├── fig5_reliability.png
│   ├── fig6_struct_error.png
│   ├── fig7_reasoning.png
│   ├── patient_graph_1_stay36605381.png
│   └── patient_graph_2_stay36200959.png
├── results
│   ├── ablations.json
│   ├── baselines.json
│   ├── test_metrics.json
│   ├── test_results.csv
│   ├── training_histories.json
│   ├── val_metrics.json
│   └── val_results.csv
├── tegot_outputs_v12
│   ├── checkpoints
│   │   ├── tegot_s1337.pt
│   │   └── tegot_s42.pt
│   ├── cohort_sub.csv
│   ├── models
│   │   └── tegot_best.pt
│   ├── plots
│   │   ├── f

In [ ]:
# =========================
# MIMIC DATA PROFILING (FAST + DECISION-FOCUSED)
# =========================

import os
import pandas as pd
import numpy as np

# ===== PATHS =====
STRUCT_PATH = "/content/mimiciv_50pct_bgot_truncated/physionet.org_extracted/files/mimiciv/3.1"
NOTE_PATH   = "/content/physionet.org/files/mimic-iv-note/2.2/note"

hosp = STRUCT_PATH + "/hosp"
icu  = STRUCT_PATH + "/icu"

print("="*80)
print("MIMIC DATA PROFILING FOR SUBSET SELECTION")
print("="*80)

# =========================
# 1. CORE TABLES
# =========================
print("\n[1] CORE COUNTS")

patients   = pd.read_csv(f"{hosp}/patients.csv")
admissions = pd.read_csv(f"{hosp}/admissions.csv")
icustays   = pd.read_csv(f"{icu}/icustays.csv")

print(f"Patients: {len(patients):,}")
print(f"Admissions: {len(admissions):,}")
print(f"ICU stays: {len(icustays):,}")

# =========================
# 2. MORTALITY DISTRIBUTION
# =========================
print("\n[2] MORTALITY")

mort_rate = admissions["hospital_expire_flag"].mean()
print(f"Mortality rate: {mort_rate:.4f}")
print("Counts:")
print(admissions["hospital_expire_flag"].value_counts())

# =========================
# 3. ICU LENGTH OF STAY
# =========================
print("\n[3] ICU LENGTH OF STAY (days)")

los = icustays["los"]
print(los.describe())

print("\nLOS buckets:")
print(pd.cut(los, bins=[0,1,3,7,14,30]).value_counts())

# =========================
# 4. DEMOGRAPHICS
# =========================
print("\n[4] DEMOGRAPHICS")

print("\nGender:")
print(patients["gender"].value_counts())

print("\nAge:")
print(patients["anchor_age"].describe())

print("\nRace (top 10):")
print(admissions["race"].value_counts().head(10))

# =========================
# 5. DIAGNOSES (KEY TASK)
# =========================
print("\n[5] DIAGNOSES DISTRIBUTION")

diag = pd.read_csv(f"{hosp}/diagnoses_icd.csv", nrows=2_000_000)

print(f"Rows sampled: {len(diag):,}")

top_icd = diag["icd_code"].value_counts().head(20)
print("\nTop 20 ICD codes:")
print(top_icd)

print("\nUnique ICD codes:", diag["icd_code"].nunique())

# =========================
# 6. LAB EVENTS
# =========================
print("\n[6] LAB EVENTS")

labs = pd.read_csv(f"{hosp}/labevents.csv", nrows=1_000_000)

print(f"Rows sampled: {len(labs):,}")
print("Unique lab items:", labs["itemid"].nunique())

print("\nTop lab tests:")
print(labs["itemid"].value_counts().head(10))

# =========================
# 7. VITALS (CHARTEVENTS)
# =========================
print("\n[7] VITALS")

charts = pd.read_csv(f"{icu}/chartevents.csv", nrows=1_000_000)

print(f"Rows sampled: {len(charts):,}")
print("Unique vital types:", charts["itemid"].nunique())

print("\nTop vital signals:")
print(charts["itemid"].value_counts().head(10))

# =========================
# 8. MEDICATIONS
# =========================
print("\n[8] MEDICATIONS")

presc = pd.read_csv(f"{hosp}/prescriptions.csv", nrows=500_000)

print(f"Rows sampled: {len(presc):,}")
print("Unique drugs:", presc["drug"].nunique())

print("\nTop drugs:")
print(presc["drug"].value_counts().head(10))

# =========================
# 9. NOTES COVERAGE
# =========================
print("\n[9] NOTES COVERAGE")

discharge = pd.read_csv(f"{NOTE_PATH}/discharge.csv", nrows=500_000)
radiology = pd.read_csv(f"{NOTE_PATH}/radiology.csv", nrows=500_000)

print("Discharge notes:", len(discharge))
print("Radiology notes:", len(radiology))

print("\nUnique admissions with discharge notes:", discharge["hadm_id"].nunique())
print("Unique admissions with radiology:", radiology["hadm_id"].nunique())

# =========================
# 10. MERGE COVERAGE (IMPORTANT)
# =========================
print("\n[10] MULTIMODAL COVERAGE")

adm_ids = set(admissions["hadm_id"].unique())
note_ids = set(discharge["hadm_id"].unique())

overlap = len(adm_ids & note_ids)
print(f"Admissions with discharge notes: {overlap}")
print(f"Coverage: {overlap / len(adm_ids):.3f}")

# =========================
# 11. RECOMMENDED SUBSET
# =========================
print("\n[11] RECOMMENDATION")

print("""
Strong subset to start with:

1. ICU stays only
2. Admissions with:
   - at least 1 discharge note
   - at least 1 lab event
   - LOS between 1 and 7 days

3. Limit to:
   - top 20 ICD codes
   - top 50 lab items
   - top 20 vital signals

4. Sample size:
   - 2000–4000 stays

This gives:
- balanced data
- manageable size
- strong signal
""")

print("\nDONE")

MIMIC DATA PROFILING FOR SUBSET SELECTION

[1] CORE COUNTS
Patients: 36,411
Admissions: 43,050
ICU stays: 45,232

[2] MORTALITY
Mortality rate: 0.1093
Counts:
hospital_expire_flag
0    38344
1     4706
Name: count, dtype: int64

[3] ICU LENGTH OF STAY (days)
count    45232.000000
mean         3.759438
std          5.480725
min          0.500000
25%          1.164334
50%          2.041088
75%          3.982491
max        226.403079
Name: los, dtype: float64

LOS buckets:
los
(1, 3]      21805
(3, 7]       9930
(0, 1]       7891
(7, 14]      3685
(14, 30]     1599
Name: count, dtype: int64

[4] DEMOGRAPHICS

Gender:
gender
M    20554
F    15857
Name: count, dtype: int64

Age:
count    36411.000000
mean        63.426465
std         16.790950
min         18.000000
25%         53.000000
50%         65.000000
75%         76.000000
max         91.000000
Name: anchor_age, dtype: float64

Race (top 10):
race
WHITE                             26774
BLACK/AFRICAN AMERICAN             3956
UNKNOWN

/tmp/ipykernel_604/256741765.py:88: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  labs = pd.read_csv(f"{hosp}/labevents.csv", nrows=1_000_000)


Rows sampled: 1,000,000
Unique lab items: 730

Top lab tests:
itemid
50971    30010
50983    29972
50902    29451
50912    28927
51006    28853
50882    28720
50868    28650
50931    28308
51221    28143
50960    27597
Name: count, dtype: int64

[7] VITALS
Rows sampled: 1,000,000
Unique vital types: 1560

Top vital signals:
itemid
227969    30888
220045    20098
220210    19837
220277    19549
220048    18270
224650    16269
220179    12513
220180    12512
220181    12499
229381     9294
Name: count, dtype: int64

[8] MEDICATIONS


/tmp/ipykernel_604/256741765.py:114: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  presc = pd.read_csv(f"{hosp}/prescriptions.csv", nrows=500_000)


Rows sampled: 500,000
Unique drugs: 1964

Top drugs:
drug
0.9% Sodium Chloride           26256
Insulin                        21387
Potassium Chloride             20769
Furosemide                     14731
Sodium Chloride 0.9%  Flush    13843
5% Dextrose                    13453
Bag                            12939
Magnesium Sulfate              12073
Metoprolol Tartrate            10896
Iso-Osmotic Dextrose            9315
Name: count, dtype: int64

[9] NOTES COVERAGE
Discharge notes: 331793
Radiology notes: 500000

Unique admissions with discharge notes: 331793
Unique admissions with radiology: 66439

[10] MULTIMODAL COVERAGE
Admissions with discharge notes: 32991
Coverage: 0.766

[11] RECOMMENDATION

Strong subset to start with:

1. ICU stays only
2. Admissions with:
   - at least 1 discharge note
   - at least 1 lab event
   - LOS between 1 and 7 days

3. Limit to:
   - top 20 ICD codes
   - top 50 lab items
   - top 20 vital signals

4. Sample size:
   - 2000–4000 stays

This give

In [1]:
#!/usr/bin/env python3

import subprocess, sys, os, json, warnings, time, gc, math, logging, copy
from collections import Counter, defaultdict
from dataclasses import dataclass, field

# --- Auto-install deps ---
def ensure(pkg):
    try:
        __import__(pkg.split("==")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("torch_geometric")
ensure("torch_scatter")
for p in ["huggingface_hub", "transformers", "peft", "safetensors", "accelerate"]:
    ensure(p)

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import networkx as nx
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.cuda.amp import autocast, GradScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss, roc_curve)
from scipy.stats import spearmanr

from torch_geometric.nn import GATv2Conv
from torch_scatter import scatter_softmax, scatter_add
from huggingface_hub import create_repo, HfApi, snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- Setup Logging ---
plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore")
for h in logging.root.handlers[:]: logging.root.removeHandler(h)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S", force=True)
log = logging.getLogger("TEGoT_v16")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
@dataclass
class Config:
    bundle_dir: str = "/content/drive/MyDrive/Papers/Nature/tegot_bundles_v16"
    output_dir: str = "/content/drive/MyDrive/Papers/Nature/tegot_outputs_v16"
    hf_repo: str = "GOVINDFROM/tegot-v16-eiml-icml2026"
    push_to_hf: bool = True

    # BioMistral LoRA Settings
    biomistral_repo: str = "GOVINDFROM/BioMistral-7B-MIMIC-Notes"
    biomistral_base: str = "BioMistral/BioMistral-7B"

    # Architecture
    note_embed_dim: int = 384
    biomistral_dim: int = 4096
    hidden_dim: int = 128
    num_heads: int = 4
    num_gat_layers: int = 2
    graph_embed_dim: int = 256
    n_obs: int = 4; n_hyp: int = 2; n_test_: int = 2; n_rev: int = 1; n_dec: int = 1
    reasoning_rounds: int = 2
    logit_clamp: float = 5.0
    gumbel_tau_start: float = 1.0; gumbel_tau_end: float = 0.5

    # Regularization & KL
    dropout: float = 0.5
    mc_dropout: float = 0.15
    weight_decay: float = 5e-3
    kl_beta_max: float = 0.01
    kl_cycle_length: int = 5
    kl_min_per_edge: float = 0.02
    logit_reg_weight: float = 0.001

    # Training
    lr: float = 1e-5
    batch_size: int = 16
    grad_accum: int = 2
    num_epochs: int = 15
    patience: int = 7
    max_grad_norm: float = 0.5
    warmup_frac: float = 0.15
    use_amp: bool = True
    pos_weight_cap: float = 3.0
    lambda_calib: float = 0.03

    # Skip connection
    skip_gate_init: float = -2.0
    bottleneck_ratio: int = 2

    # Eval
    mc_K_dropout: int = 5
    mc_K_gumbel: int = 10
    mc_eval_tau: float = 1.0
    val_calib_frac: float = 0.2
    baseline_epochs: int = 10
    ablation_epochs: int = 10
    n_ensemble: int = 3
    seeds: list = field(default_factory=lambda: [42, 1337])
    seed: int = 42

cfg = Config()
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    log.info(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
    cfg.use_amp = False

for d in [cfg.output_dir, f"{cfg.output_dir}/plots", f"{cfg.output_dir}/models",
          f"{cfg.output_dir}/results", f"{cfg.output_dir}/checkpoints", f"{cfg.output_dir}/raw"]:
    os.makedirs(d, exist_ok=True)

def flush():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def save_json(obj, path):
    def default(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return str(o)
    with open(path, "w") as f: json.dump(obj, f, indent=2, default=default)

# ══════════════════════════════════════════════════════════════════════════════
# LOAD BIOMISTRAL-7B + LORA
# ══════════════════════════════════════════════════════════════════════════════
log.info("Downloading and initializing BioMistral-7B with MIMIC-IV LoRA...")
local_path = snapshot_download(repo_id=cfg.biomistral_repo)
llm_tokenizer = AutoTokenizer.from_pretrained(cfg.biomistral_base, trust_remote_code=True)
llm_base_model = AutoModelForCausalLM.from_pretrained(
    cfg.biomistral_base,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)
adapter_path = os.path.join(local_path, "best_model")
llm_model = PeftModel.from_pretrained(llm_base_model, adapter_path)
llm_model.eval()
log.info("✓ BioMistral initialized successfully.")

# ══════════════════════════════════════════════════════════════════════════════
# LOAD BUNDLES + METADATA
# ══════════════════════════════════════════════════════════════════════════════
meta_path = f"{cfg.bundle_dir}/meta.json"
assert os.path.exists(meta_path), f"Missing {meta_path} — run bundle builder first!"
with open(meta_path) as f: meta = json.load(f)

n_dx_classes = meta['n_dx_classes']
NOTE_DIM = meta['note_embed_dim']
log.info(f"Bundle meta: n_dx={n_dx_classes}, precomputed_note_dim={NOTE_DIM}")

cohort = pd.read_csv(f"{cfg.bundle_dir}/cohort.csv")
mort_rate = meta['mort_rate_train']
mort_pw = float(min((1 - mort_rate) / max(mort_rate, 0.01), cfg.pos_weight_cap))
CONST_PRED = mort_rate
log.info(f"Mortality rate: {mort_rate:.3f}, pos_weight: {mort_pw:.2f}")

class DS(Dataset):
    def __init__(self, path, desc=""):
        self.bundle = torch.load(path, map_location='cpu', weights_only=False)
        self.stay_ids = list(self.bundle.keys())
        log.info(f"  {desc}: {len(self.stay_ids)} graphs")
    def __len__(self): return len(self.stay_ids)
    def __getitem__(self, idx): return self.bundle[self.stay_ids[idx]]

train_ds = DS(f"{cfg.bundle_dir}/train_bundle.pt", "Train")
val_full = DS(f"{cfg.bundle_dir}/val_bundle.pt",   "Val")
test_ds  = DS(f"{cfg.bundle_dir}/test_bundle.pt",  "Test")

rng = np.random.RandomState(cfg.seed)
perm = rng.permutation(len(val_full.stay_ids))
nc = int(cfg.val_calib_frac * len(perm))
val_calib_ids = [val_full.stay_ids[i] for i in perm[:nc]]
val_model_ids = [val_full.stay_ids[i] for i in perm[nc:]]

class SubDS(Dataset):
    def __init__(self, ids, graphs, desc=""):
        self.stay_ids = ids; self.graphs = graphs
        log.info(f"  {desc}: {len(ids)}")
    def __len__(self): return len(self.stay_ids)
    def __getitem__(self, idx): return self.graphs[self.stay_ids[idx]]

val_calib = SubDS(val_calib_ids, val_full.bundle, "Val-Calib")
val_model = SubDS(val_model_ids, val_full.bundle, "Val-Model")
n_train, n_val, n_test = len(train_ds), len(val_model), len(test_ds)

# ══════════════════════════════════════════════════════════════════════════════
# MODEL DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════
FEAT_DIMS = {"patient": 3, "diagnosis": 7, "lab": 3, "vital": 2, "medication": 4, "procedure": 3, "io": 2, "micro": 2}

class MCDrop(nn.Module):
    def __init__(self, p=0.15): super().__init__(); self.p = p
    def forward(self, x): return F.dropout(x, self.p, training=True)

class TemporalEnc(nn.Module):
    def __init__(self, d, nf=16):
        super().__init__()
        self.f = nn.Parameter(torch.randn(nf)*0.1)
        self.proj = nn.Linear(nf*2+2, d)
    def forward(self, t):
        t = t.unsqueeze(-1).float()
        f = self.f.exp().clamp(max=100)
        sc = torch.cat([torch.sin(t*f), torch.cos(t*f)], -1)
        ex = torch.cat([t/720., torch.log1p(t.abs())], -1)
        return self.proj(torch.cat([sc, ex], -1))

class HeteroEncoder(nn.Module):
    def __init__(self, fdims, h, heads, layers, edim, drop, mcdrop):
        super().__init__()
        self.projs = nn.ModuleDict({n: nn.Sequential(nn.Linear(d,h),nn.LayerNorm(h)) for n,d in fdims.items()})
        self.absent = nn.ParameterDict({n: nn.Parameter(torch.randn(h)*0.02) for n in fdims})
        self.te = TemporalEnc(h)
        self.h = h; self.edim = edim
        self.gats = nn.ModuleList([GATv2Conv(h,h//heads,heads=heads,dropout=drop,add_self_loops=False) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(h) for _ in range(layers)])
        self.drop = nn.Dropout(drop)
        self.mcds = nn.ModuleList([MCDrop(mcdrop) for _ in range(layers)])
        self.pg = nn.Linear(h,1); self.pp = nn.Linear(h,edim); self.mcp = MCDrop(mcdrop)

    def forward(self, xd, eid, td):
        h = {}
        for nt, x in xd.items():
            if nt not in self.projs: continue
            if x.shape[0]==1 and x.abs().sum()<1e-8 and nt in self.absent:
                h[nt] = self.absent[nt].unsqueeze(0); continue
            h[nt] = self.projs[nt](x)
            if nt in td and td[nt].numel()>0: h[nt] = h[nt] + self.te(td[nt])
        for li in range(len(self.gats)):
            gat,nm,mcd = self.gats[li],self.norms[li],self.mcds[li]
            upd = defaultdict(list)
            for et, ei in eid.items():
                st,_,dt = et
                if st not in h or dt not in h: continue
                sh,dh = h[st],h[dt]; ns,nd = sh.shape[0],dh.shape[0]
                c = torch.cat([sh,dh],0)
                se = ei.clone().to(c.device); se[1] = se[1]+ns
                v = (se[0]<c.shape[0])&(se[1]<c.shape[0])
                if v.sum()==0: continue
                se = se[:,v]
                try:
                    out = gat(c, se); upd[dt].append(out[ns:ns+nd])
                except: continue
            for nt in h:
                if nt in upd and upd[nt]:
                    agg = torch.stack(upd[nt]).mean(0)
                    h[nt] = nm(h[nt] + self.drop(agg)); h[nt] = mcd(h[nt])
        vals = [v for v in h.values() if v.numel()>0]
        if not vals: return h, torch.zeros(self.edim, device=next(self.parameters()).device)
        ah = torch.cat(vals,0)
        g = torch.sigmoid(self.pg(ah))
        return h, self.mcp(self.pp((g*ah).sum(0)))

class BioMistralNoteEncoder(nn.Module):
    def __init__(self, biomistral_dim, note_embed_dim, drop=0.1):
        super().__init__()
        # Linear projection to map 4096-dim BioMistral features to 384-dim TEGoT space
        self.proj = nn.Sequential(
            nn.Linear(biomistral_dim, 1024),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(1024, note_embed_dim),
            nn.LayerNorm(note_embed_dim)
        )

    def process_text(self, text_list):
        """Passes raw text through the loaded BioMistral PeftModel to get hidden states."""
        with torch.no_grad():
            inputs = llm_tokenizer(text_list, return_tensors="pt", padding=True, truncation=True, max_length=512).to(llm_model.device)
            outputs = llm_model(**inputs, output_hidden_states=True)
            # Take mean pooling of the last hidden state
            hidden_states = outputs.hidden_states[-1].mean(dim=1)
        return hidden_states.to(device)

    def forward(self, notes_input, is_text=False):
        if is_text:
            notes_input = self.process_text(notes_input)
        return self.proj(notes_input)

class NoteFusion(nn.Module):
    def __init__(self, edim, ndim, nh=4, drop=0.1):
        super().__init__()
        self.biomistral_encoder = BioMistralNoteEncoder(cfg.biomistral_dim, ndim, drop)
        # Using Cross-Attention: Graph (hG) queries Note features (hn)
        self.ca = nn.MultiheadAttention(edim,nh,dropout=drop,batch_first=True)
        self.cn = nn.LayerNorm(edim)
        self.gp = nn.Linear(edim*2,edim)
        nn.init.constant_(self.gp.bias, -2.2)

    def forward(self, hG, nx, is_text=False):
        if nx is None: return hG
        if not is_text and nx.shape[0]==1 and nx.abs().sum()<1e-8: return hG

        hn = self.biomistral_encoder(nx, is_text)
        q = hG.unsqueeze(0).unsqueeze(0); kv = hn.unsqueeze(0)
        ha,_ = self.ca(q,kv,kv)
        ha = self.cn(ha.squeeze(0).squeeze(0))
        g = torch.sigmoid(self.gp(torch.cat([hG,ha],-1)))
        return hG + g*ha

class VarReasoning(nn.Module):
    def __init__(self, h, edim, no=4, nh=2, nt=2, nr=1, nd=1, mkl=0.02, lc=5.0):
        super().__init__()
        self.n_total = no+nh+nt+nr+nd
        self.node_types = ["obs"]*no+["hyp"]*nh+["test"]*nt+["rev"]*nr+["dec"]*nd
        self.mkl = mkl; self.lc = lc
        self.tmpl = nn.Parameter(torch.randn(self.n_total,h)*0.02)
        self.ca = nn.MultiheadAttention(h,4,dropout=0.1,batch_first=True)
        self.cn = nn.LayerNorm(h)
        self.cp = nn.Sequential(nn.Linear(edim,h),nn.GELU(),nn.LayerNorm(h))
        self.em = nn.Sequential(nn.Linear(h*2+edim,h),nn.GELU(),nn.Dropout(0.1),nn.Linear(h,1))
        nn.init.normal_(self.em[-1].weight, std=0.02); nn.init.zeros_(self.em[-1].bias)
        rngs = {}; off = 0
        for g,c in [("obs",no),("hyp",nh),("test",nt),("rev",nr),("dec",nd)]:
            rngs[g] = (off,off+c); off += c
        pm = {("obs","hyp"):0.4,("hyp","test"):0.5,("test","rev"):0.6,("rev","dec"):0.7,
              ("obs","test"):0.15,("obs","dec"):0.05,("hyp","rev"):0.2,("hyp","dec"):0.1,("test","dec"):0.3}
        es,ps = [],[]
        for (st,dt),p in pm.items():
            for s in range(*rngs[st]):
                for d in range(*rngs[dt]):
                    es.append((s,d)); ps.append(p)
        self.register_buffer("cand", torch.tensor(es,dtype=torch.long))
        self.register_buffer("prior", torch.tensor(ps,dtype=torch.float))

    def forward(self, hd, hf, tau=1.0):
        z = self.tmpl.unsqueeze(0)
        vs = [v for v in hd.values() if v.numel()>0]
        ctx = self.cp(hf).unsqueeze(0)
        kv = (torch.cat([torch.cat(vs,0),ctx],0) if vs else ctx).unsqueeze(0)
        za,_ = self.ca(z,kv,kv)
        z = self.cn(z+za).squeeze(0)
        s,d = self.cand[:,0],self.cand[:,1]
        h2 = hf.unsqueeze(0).expand(len(s),-1)
        logits = self.em(torch.cat([z[s],z[d],h2],-1)).squeeze(-1)
        logits = self.lc * torch.tanh(logits/self.lc)
        q = torch.sigmoid(logits).clamp(1e-6,1-1e-6)
        u = torch.rand_like(logits).clamp(1e-6,1-1e-6)
        gn = torch.log(u)-torch.log(1-u)
        sampled = torch.sigmoid((logits+gn)/max(tau,0.1))
        p = self.prior.clamp(1e-6,1-1e-6)
        kpe = q*(q.log()-p.log())+(1-q)*((1-q).log()-(1-p).log())
        kl = torch.clamp(kpe, min=self.mkl).sum()
        lr = (logits.abs()-(self.lc*0.8)).clamp(min=0).pow(2).mean()
        return z, q, sampled, kl, lr, kpe

class Decoder(nn.Module):
    def __init__(self, h, edim, ntot, nr=2, mcd=0.15, br=2):
        super().__init__()
        self.nr = nr
        self.ml = nn.Linear(h,h); self.gru = nn.GRUCell(h,h)
        self.nm = nn.LayerNorm(h); self.mcd = MCDrop(mcd)
        b = edim//br
        self.op = nn.Sequential(nn.Linear(h,b),nn.GELU(),nn.Linear(b,edim),
                                 nn.LayerNorm(edim),nn.GELU(),nn.Dropout(0.1),nn.Linear(edim,edim))
    def forward(self, z, samp, ce):
        h = z; s,d = ce[:,0],ce[:,1]; nn_ = h.shape[0]
        for _ in range(self.nr):
            raw = self.ml(h[s])
            att = scatter_softmax(samp, d, dim=0)
            agg = scatter_add(raw*att.unsqueeze(-1), d, dim=0, dim_size=nn_)
            h = self.nm(self.gru(agg, h))
        return self.op(self.mcd(h.mean(0)))

class Heads(nn.Module):
    def __init__(self, edim, mcd=0.15):
        super().__init__()
        self.mort = nn.Sequential(nn.Linear(edim,128),nn.GELU(),MCDrop(mcd),nn.Linear(128,32),nn.GELU(),nn.Linear(32,1))
    def forward(self, h):
        if h.dim()==1: h=h.unsqueeze(0)
        return {"mortality": self.mort(h).view(-1)}

def ece(probs, labels, nb=10):
    b = torch.linspace(0,1,nb+1,device=probs.device); e = torch.tensor(0.0,device=probs.device)
    tot = probs.shape[0]
    if tot==0: return e
    for i in range(nb):
        m = (probs>=b[i])&(probs<b[i+1])
        if i==nb-1: m = m|(probs==b[i+1])
        n = m.sum().float()
        if n>0: e += (n/tot)*(probs[m].mean()-labels[m].float().mean()).abs()
    return e

class Loss(nn.Module):
    def __init__(self, pw, dev, lc=0.03, lrw=0.001):
        super().__init__()
        self.pw = torch.tensor([pw],device=dev)
        self.lc = lc; self.lrw = lrw
    def forward(self, preds, tgt, kl, lr, beta=0.01):
        logits = torch.clamp(preds["mortality"], -20.0, 20.0)
        lm = F.binary_cross_entropy_with_logits(logits, tgt["mortality"], pos_weight=self.pw)
        mp = torch.sigmoid(preds["mortality"]).detach()
        ec = ece(mp, tgt["mortality"])
        tot = lm + beta*kl + self.lrw*lr + self.lc*ec
        return tot, {"mort":lm.item(),"kl":kl.item(),"ece":ec.item(),"total":tot.item()}

class TEGoT(nn.Module):
    def __init__(self, c):
        super().__init__()
        nr = c.n_obs+c.n_hyp+c.n_test_+c.n_rev+c.n_dec
        self.enc = HeteroEncoder(FEAT_DIMS,c.hidden_dim,c.num_heads,c.num_gat_layers,c.graph_embed_dim,c.dropout,c.mc_dropout)
        self.nf = NoteFusion(c.graph_embed_dim,c.note_embed_dim)
        self.reas = VarReasoning(c.hidden_dim,c.graph_embed_dim,c.n_obs,c.n_hyp,c.n_test_,c.n_rev,c.n_dec,c.kl_min_per_edge,c.logit_clamp)
        self.dec = Decoder(c.hidden_dim,c.graph_embed_dim,nr,c.reasoning_rounds,c.mc_dropout,c.bottleneck_ratio)
        self.heads = Heads(c.graph_embed_dim,c.mc_dropout)
        self.sg = nn.Parameter(torch.tensor(c.skip_gate_init))
        self.lt = nn.Parameter(torch.tensor(0.0))
        log.info(f"  TEGoT params: {sum(p.numel() for p in self.parameters()):,}")

    def _unpack(self, data, zt=False):
        xd,td,eid = {},{},{}
        for nt in data.node_types:
            if nt=="note": continue
            xd[nt] = data[nt].x.to(device)
            if hasattr(data[nt],"t") and not zt: td[nt] = data[nt].t.to(device)
        for et in data.edge_types:
            s,r,d = et
            if s=="note" or d=="note" or "xmodal" in r: continue
            eid[et] = data[et].edge_index.to(device)
        return xd,td,eid

    def _notes(self, data):
        # Allow dynamic text processing or fallback to precomputed features
        is_text = hasattr(data['note'], 'text')
        nx = data['note'].text if is_text else data['note'].x.to(device)
        return nx, is_text

    def forward(self, data, tau=1.0, zt=False, mn=False):
        xd,td,eid = self._unpack(data,zt)
        hd,hG = self.enc(xd,eid,td)
        nx, is_text = self._notes(data)
        hf = hG if mn else self.nf(hG, nx, is_text=is_text)
        z,q,samp,kl,lr,kpe = self.reas(hd,hf,tau)
        hr = self.dec(z,samp,self.reas.cand)
        gate = torch.sigmoid(self.sg)
        hfin = hr + gate*hf
        preds = self.heads(hfin)
        return preds,kl,lr,q,samp,kpe

    @torch.no_grad()
    def mc_factored(self, data, Kd=5, Kg=10, tau=1.0, mn=False, zt=False):
        T = self.lt.exp().clamp(0.5,5.0).item()
        all_p = np.zeros((Kd,Kg))
        all_q, all_e = [], []
        for i in range(Kd):
            for j in range(Kg):
                p,_,_,q,s,_ = self.forward(data,tau,zt=zt,mn=mn)
                all_p[i,j] = torch.sigmoid(torch.tensor(p["mortality"].item()/T)).item()
                if i==0:
                    all_e.append(s.detach().cpu())
                    all_q.append(q.detach().cpu())
        mp = float(all_p.mean())
        def H(p):
            p=np.clip(p,1e-8,1-1e-8); return -(p*np.log(p)+(1-p)*np.log(1-p))
        ut = H(mp); ua = float(np.mean([H(x) for x in all_p.flatten()]))
        ue = float(np.var(all_p.mean(1)))   # var of dropout-means
        us = float(np.var(all_p.mean(0)))   # var of Gumbel-means
        nn_ = sum(data[nt].x.shape[0] for nt in data.node_types if nt!="note")
        return {"mortality_mean":mp,"u_total":ut,"u_aleatoric":ua,"u_epistemic":ue,"u_structural":us,
                "mort_std":float(all_p.flatten().std()),"q_var":float(torch.stack(all_q).var(0).mean()) if all_q else 0,
                "edge_var":float(torch.stack(all_e).var(0).mean()) if all_e else 0,
                "n_graph_nodes":nn_,"skip_gate":float(torch.sigmoid(self.sg).item())}

# ══════════════════════════════════════════════════════════════════════════════
# TRAINING INFRASTRUCTURE
# ══════════════════════════════════════════════════════════════════════════════
def get_lr(step, warmup, total, base):
    if step<warmup: return base*(step+1)/warmup
    return 0.5*base*(1+math.cos(math.pi*(step-warmup)/max(1,total-warmup)))

def get_beta(ep, cyc, bmax):
    t = (ep%cyc)/cyc; return bmax*min(2*t,1.0)

def learn_temp(mdl, cds):
    mdl.eval(); ls,ys = [],[]
    with torch.no_grad():
        for i in range(len(cds)):
            try:
                d=cds[i]; p,_,_,_,_,_ = mdl(d,tau=0.5)
                ls.append(p["mortality"].item()); ys.append(d.y_mortality.item())
            except: continue
    if len(ls)<10 or len(set(ys))<2:
        with torch.no_grad(): mdl.lt.fill_(0.0); return 1.0
    lt=torch.tensor(ls,device=device); yt=torch.tensor(ys,device=device)
    best_n,best_T = float("inf"),1.0
    for T in np.arange(0.3,5.0,0.05):
        n=F.binary_cross_entropy_with_logits(lt/T,yt).item()
        if n<best_n: best_n,best_T = n,float(T)
    with torch.no_grad(): mdl.lt.fill_(math.log(best_T))
    log.info(f"  T={best_T:.2f} (NLL={best_n:.4f})")
    flush(); return best_T

def train_seed(seed, cls=TEGoT, nep=None, label="TEGoT", ckpt=None):
    nep = nep or cfg.num_epochs
    if ckpt and os.path.exists(ckpt):
        log.info(f"  [{label} s{seed}] Loading checkpoint")
        mdl = cls(cfg).to(device)
        mdl.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True),strict=False)
        learn_temp(mdl, val_calib); return mdl, {}, 0

    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    mdl = cls(cfg).to(device)
    crit = Loss(mort_pw,device,cfg.lambda_calib,cfg.logit_reg_weight)
    mp = [p for n,p in mdl.named_parameters() if n!="lt"]
    opt = torch.optim.AdamW(mp,lr=cfg.lr,weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=cfg.use_amp)
    spe = n_train//cfg.batch_size+1
    total_st = spe*nep; warmup = int(cfg.warmup_frac*total_st)
    best_vl,best_st,pat = float("inf"),None,0
    hist = defaultdict(list); t0 = time.time(); gs = 0

    for ep in range(nep):
        mdl.train(); losses,kls,mls = [],[],[]
        tau = cfg.gumbel_tau_start+(cfg.gumbel_tau_end-cfg.gumbel_tau_start)*ep/max(nep-1,1)
        beta = get_beta(ep,cfg.kl_cycle_length,cfg.kl_beta_max)
        perm = np.random.permutation(n_train); gnorms = []; t_ep = time.time()
        opt.zero_grad(); acc = 0
        pbar = tqdm(range(0,len(perm),cfg.batch_size), desc=f"[{label} s{seed}] ep{ep+1}/{nep}", mininterval=2, leave=False)
        for bi in pbar:
            lr_now = get_lr(gs,warmup,total_st,cfg.lr)
            for g in opt.param_groups: g['lr'] = lr_now
            bl = []
            for idx in perm[bi:bi+cfg.batch_size]:
                try:
                    d = train_ds[idx]
                    with autocast(enabled=cfg.use_amp, dtype=torch.bfloat16 if cfg.use_amp and torch.cuda.is_bf16_supported() else torch.float16):
                        p, kl, lr, _, _, _ = mdl(d, tau=tau)
                        tgt = {"mortality": d.y_mortality.view(1).to(device)}
                        loss, det = crit(p, tgt, kl, lr, beta=beta)
                    if not (torch.isnan(loss) or torch.isinf(loss)):
                        bl.append(loss); losses.append(det["total"]); kls.append(det["kl"]); mls.append(det["mort"])
                except Exception as e: continue
            if not bl: gs+=1; continue
            tot = torch.stack(bl).mean()/cfg.grad_accum
            scaler.scale(tot).backward(); acc += 1
            if acc%cfg.grad_accum==0:
                scaler.unscale_(opt); gn = nn.utils.clip_grad_norm_(mp,cfg.max_grad_norm).item()
                gnorms.append(gn)
                scaler.step(opt); scaler.update(); opt.zero_grad()
            gs += 1
            if len(losses)%60==0: pbar.set_postfix({"l":f"{np.mean(losses[-60:]):.3f}","β":f"{beta:.4f}"})
        if acc%cfg.grad_accum!=0:
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(mp,cfg.max_grad_norm)
            scaler.step(opt); scaler.update(); opt.zero_grad()
        pbar.close()

        mdl.eval(); vls = []
        with torch.no_grad():
            for i in range(n_val):
                try:
                    d = val_model[i]
                    with autocast(enabled=cfg.use_amp, dtype=torch.bfloat16 if cfg.use_amp and torch.cuda.is_bf16_supported() else torch.float16):
                        p, kl, lr, _, _, _ = mdl(d, tau=0.5)
                        tgt = {"mortality": d.y_mortality.view(1).to(device)}
                        vl, _ = crit(p, tgt, kl, lr, beta=beta)
                    if not (torch.isnan(vl) or torch.isinf(vl)): vls.append(vl.item())
                except: continue
        flush()

        at, av = float(np.mean(losses)) if losses else float("nan"), float(np.mean(vls)) if vls else float("nan")
        sg = float(torch.sigmoid(mdl.sg).item())
        hist["epoch"].append(ep+1); hist["tr"].append(at); hist["val"].append(av); hist["kl"].append(float(np.mean(kls)) if kls else 0)
        hist["beta"].append(beta); hist["sg"].append(sg); hist["gn"].append(max(gnorms) if gnorms else 0)
        hist["t"].append(time.time()-t_ep)

        improved = not math.isnan(av) and av<best_vl
        if improved:
            best_vl=av; best_st={k:v.detach().cpu().clone() for k,v in mdl.state_dict().items()}; pat=0
        else: pat+=1
        tag = "★" if improved else f"({pat}/{cfg.patience})"
        log.info(f"  [{label} s{seed}] ep{ep+1}/{nep} | tr={at:.4f} val={av:.4f} | β={beta:.4f} kl={float(np.mean(kls)) if kls else 0:.3f} gn={max(gnorms) if gnorms else 0:.1f} sg={sg:.3f} | {time.time()-t_ep:.0f}s | {tag}")
        if pat>=cfg.patience: log.info(f"  [{label} s{seed}] early stop @ ep{ep+1}"); break

    if best_st: mdl.load_state_dict(best_st)
    log.info(f"  [{label} s{seed}] done {(time.time()-t0)/60:.1f}min, best_val={best_vl:.4f}")
    learn_temp(mdl, val_calib)
    if ckpt: torch.save(mdl.state_dict(), ckpt)
    flush()
    return mdl, hist, best_vl

# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION UTILITIES
# ══════════════════════════════════════════════════════════════════════════════
def bci(y,p,fn,nb=1000):
    y,p=np.asarray(y),np.asarray(p); n=len(y)
    if n<5 or len(np.unique(y))<2: return None,None,None
    rng=np.random.RandomState(42); sc=[]
    for _ in range(nb):
        idx=rng.randint(0,n,n)
        if len(np.unique(y[idx]))<2: continue
        try: sc.append(fn(y[idx],p[idx]))
        except: continue
    if not sc: return None,None,None
    s=np.array(sc)
    return float(s.mean()),float(np.quantile(s,0.025)),float(np.quantile(s,0.975))

def bpaired(y,p1,p2,fn,nb=1000):
    y,p1,p2=np.asarray(y),np.asarray(p1),np.asarray(p2); n=len(y)
    if n<5 or len(np.unique(y))<2: return None,None,None,None
    rng=np.random.RandomState(42); ds=[]
    for _ in range(nb):
        i=rng.randint(0,n,n)
        if len(np.unique(y[i]))<2: continue
        try: ds.append(fn(y[i],p1[i])-fn(y[i],p2[i]))
        except: continue
    if not ds: return None,None,None,None
    d=np.array(ds); pv=2*min((d<=0).mean(),(d>=0).mean())
    return float(d.mean()),float(np.quantile(d,0.025)),float(np.quantile(d,0.975)),float(pv)

def eval_models(models, dataset, desc="Eval"):
    rows = []
    for mi,m in enumerate(models):
        m.eval()
        for j in tqdm(range(len(dataset)),desc=f"{desc} m{mi+1}/{len(models)}",leave=False):
            try:
                d=dataset[j]
                mc=m.mc_factored(d,Kd=cfg.mc_K_dropout,Kg=cfg.mc_K_gumbel,tau=cfg.mc_eval_tau)
                rows.append({"stay_id":dataset.stay_ids[j],"y_mort":d.y_mortality.item(),
                             "p_mort":mc["mortality_mean"],"u_total":mc["u_total"],
                             "u_aleatoric":mc["u_aleatoric"],"u_epistemic":mc["u_epistemic"],
                             "u_structural":mc["u_structural"],"mort_std":mc["mort_std"],
                             "q_var":mc["q_var"],"edge_var":mc["edge_var"],
                             "n_nodes":mc["n_graph_nodes"],"sg":mc["skip_gate"],"seed":mi})
            except: continue
            if (j+1)%200==0: flush()
        flush()
    return pd.DataFrame(rows)

def avg_stay(df):
    return df.groupby("stay_id").agg({
        "y_mort":"first","p_mort":"mean","u_total":"mean","u_aleatoric":"mean",
        "u_epistemic":"mean","u_structural":"mean","mort_std":"mean","q_var":"mean",
        "edge_var":"mean","n_nodes":"first","sg":"first"}).reset_index()

def metrics(df, cp=None):
    m={"n":len(df)}
    if len(df)<5 or len(np.unique(df.y_mort))<2: return m
    y,p=df.y_mort.values,df.p_mort.values
    for k,fn in [("auroc",roc_auc_score),("auprc",average_precision_score),("brier",brier_score_loss)]:
        mn,lo,hi = bci(y,p,fn); m[k]={"mean":mn,"ci_lo":lo,"ci_hi":hi}
    if cp is not None:
        bm=np.mean((y-p)**2); bc=np.mean((y-cp)**2)
        m["brier_skill"]=1-bm/bc if bc>1e-12 else None
    errors = np.abs(y-p)
    for uc in ["u_structural","u_epistemic","u_aleatoric","u_total"]:
        if uc in df.columns and np.std(df[uc])>0:
            rho,pv = spearmanr(df[uc],errors)
            m[f"{uc}_rho"]={"rho":float(rho),"p":float(pv)}
    if "u_structural" in df.columns and "n_nodes" in df.columns:
        us=df.u_structural.values; nn_=df.n_nodes.values
        if np.std(nn_)>0 and np.std(us)>0:
            from numpy.polynomial import polynomial as P
            coef = P.polyfit(nn_,us,1); us_resid = us - P.polyval(nn_,coef)
            if np.std(us_resid)>0:
                rho,pv = spearmanr(us_resid, errors)
                m["u_struct_resid_rho"]={"rho":float(rho),"p":float(pv)}
    for c in ["u_total","u_epistemic","u_structural","u_aleatoric","mort_std","q_var","edge_var"]:
        if c in df.columns: m[f"avg_{c}"]=float(df[c].mean())
    return m

def youden(y,p):
    fpr,tpr,thr=roc_curve(y,p)
    return float(thr[np.argmax(tpr-fpr)])

def clin(y,p,thr):
    y=np.asarray(y).astype(int); pr=(np.asarray(p)>=thr).astype(int)
    tp=int(((pr==1)&(y==1)).sum()); fp=int(((pr==1)&(y==0)).sum())
    tn=int(((pr==0)&(y==0)).sum()); fn=int(((pr==0)&(y==1)).sum())
    return {"thr":float(thr),"tp":tp,"fp":fp,"tn":tn,"fn":fn,
            "sens":tp/max(tp+fn,1),"spec":tn/max(tn+fp,1),"ppv":tp/max(tp+fp,1),"npv":tn/max(tn+fn,1)}

def extract_tab(data):
    feats = []
    for nt in ['patient','diagnosis','lab','vital','medication','procedure','io','micro']:
        d = FEAT_DIMS[nt]
        if nt in data.node_types:
            x=data[nt].x
            if x.shape[0]>0 and x.abs().sum()>1e-8:
                feats.extend([x.mean(0).numpy(),x.max(0)[0].numpy(),
                              x.std(0).numpy() if x.shape[0]>1 else np.zeros(d),
                              np.array([x.shape[0]],dtype=np.float32)])
            else: feats.extend([np.zeros(d)]*3+[np.zeros(1)])
        else: feats.extend([np.zeros(d)]*3+[np.zeros(1)])
    if 'note' in data.node_types and not hasattr(data['note'], 'text'):
        nx_=data['note'].x
        feats.append(nx_.mean(0).numpy() if nx_.shape[0]>0 and nx_.abs().sum()>1e-8 else np.zeros(cfg.note_embed_dim))
    else: feats.append(np.zeros(cfg.note_embed_dim))
    return np.concatenate([f.astype(np.float32) for f in feats])

def build_tab(ds,desc):
    X,y=[],[]
    for i in tqdm(range(len(ds)),desc=desc,leave=False):
        try:
            d=ds[i]; X.append(extract_tab(d)); y.append(d.y_mortality.item())
        except: continue
    return np.nan_to_num(np.array(X,dtype=np.float32)),np.array(y,dtype=np.int32)

# ══════════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70 + "\nSTEP 1: TABULAR BASELINES (LR + MLP)\n" + "="*70)

baselines = []
y_sample = np.array([test_ds[i].y_mortality.item() for i in range(min(100, len(test_ds)))])
baselines.append({"name": "Constant", "auroc": 0.5, "brier": float(np.mean((y_sample - CONST_PRED) ** 2))})

X_tr,y_tr = build_tab(train_ds,"tab-train"); X_va,y_va = build_tab(val_model,"tab-val"); X_te,y_te = build_tab(test_ds,"tab-test")
sc = StandardScaler(); X_trs=sc.fit_transform(X_tr); X_vas=sc.transform(X_va); X_tes=sc.transform(X_te)
log.info(f"Tabular: X_tr={X_tr.shape}, X_te={X_te.shape}")

# LR
best_C,best_auc,best_lr = 1.0,0.0,None
for C in [0.001,0.01,0.1,1.0,10.0]:
    lr_m=LogisticRegression(C=C,max_iter=500,class_weight='balanced',solver='liblinear',random_state=42)
    try:
        lr_m.fit(X_trs,y_tr)
        pv=lr_m.predict_proba(X_vas)[:,1]
        a=roc_auc_score(y_va,pv) if len(np.unique(y_va))>=2 else 0.5
        if a>best_auc: best_C,best_auc,best_lr=C,a,lr_m
    except: continue
p_lr = best_lr.predict_proba(X_tes)[:,1]
a,lo,hi = bci(y_te,p_lr,roc_auc_score)
baselines.append({"name":"LR","C":best_C,"auroc":a,"ci":[lo,hi],"brier":float(brier_score_loss(y_te,p_lr))})
np.save(f"{cfg.output_dir}/raw/p_lr.npy",p_lr); log.info(f"  LR: AUROC={a:.4f} [{lo:.4f},{hi:.4f}]")

# MLP
mlp=MLPClassifier(hidden_layer_sizes=(256,128,64),max_iter=100,learning_rate_init=1e-3,random_state=42,early_stopping=True,validation_fraction=0.15)
try:
    mlp.fit(X_trs,y_tr)
    p_mlp=mlp.predict_proba(X_tes)[:,1]
    a,lo,hi=bci(y_te,p_mlp,roc_auc_score)
    baselines.append({"name":"MLP","auroc":a,"ci":[lo,hi],"brier":float(brier_score_loss(y_te,p_mlp))})
    np.save(f"{cfg.output_dir}/raw/p_mlp.npy",p_mlp); log.info(f"  MLP: AUROC={a:.4f}")
except Exception as e: log.warning(f"  MLP failed: {e}")
save_json(baselines, f"{cfg.output_dir}/results/baselines_tabular.json")

print("\n" + "="*70 + "\nSTEP 2: TEGoT v16 × SEEDS\n" + "="*70)

tegot_ms, tegot_hs = [], []
for sd in cfg.seeds:
    ck=f"{cfg.output_dir}/checkpoints/tegot_s{sd}.pt"
    m,h,_ = train_seed(sd,TEGoT,label="TEGoT",ckpt=ck)
    tegot_ms.append(m); tegot_hs.append(h); flush()
torch.save(tegot_ms[0].state_dict(), f"{cfg.output_dir}/models/tegot_best.pt")
save_json({f"s{sd}":dict(h) for sd,h in zip(cfg.seeds,tegot_hs)}, f"{cfg.output_dir}/results/train_hist.json")

print("\n" + "="*70 + "\nSTEP 3: GATv2 BASELINES\n" + "="*70)

class GATv2BL(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.enc = HeteroEncoder(FEAT_DIMS,c.hidden_dim,c.num_heads,c.num_gat_layers,c.graph_embed_dim,c.dropout,0.0)
        self.mort = nn.Sequential(nn.Linear(c.graph_embed_dim,128),nn.GELU(),nn.Dropout(0.3),nn.Linear(128,1))
        self.sg = nn.Parameter(torch.tensor(-10.0)); self.lt = nn.Parameter(torch.tensor(0.0))
    def forward(self, data, **kw):
        xd,td,eid = {},{},{}
        for nt in data.node_types:
            if nt=="note": continue
            xd[nt]=data[nt].x.to(device)
            if hasattr(data[nt],"t"): td[nt]=data[nt].t.to(device)
        for et in data.edge_types:
            s,r,d=et
            if s=="note" or d=="note" or "xmodal" in r: continue
            eid[et]=data[et].edge_index.to(device)
        _,hG = self.enc(xd,eid,td)
        if hG.dim()==1: hG=hG.unsqueeze(0)
        return {"mortality":self.mort(hG).view(-1)}

gat_probs = []
for k in range(cfg.n_ensemble):
    sd=cfg.seed+k*1000
    log.info(f"  GATv2 seed={sd}")
    torch.manual_seed(sd); np.random.seed(sd)
    bl=GATv2BL(cfg).to(device)
    opt=torch.optim.AdamW(bl.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay)
    cr=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([mort_pw],device=device))
    scaler=GradScaler(enabled=cfg.use_amp)
    for ep in range(cfg.baseline_epochs):
        bl.train(); opt.zero_grad(); perm=np.random.permutation(n_train)
        for bi in range(0,len(perm),cfg.batch_size):
            bls=[]
            for idx in perm[bi:bi+cfg.batch_size]:
                try:
                    d=train_ds[idx]
                    with autocast(enabled=cfg.use_amp, dtype=torch.bfloat16 if cfg.use_amp and torch.cuda.is_bf16_supported() else torch.float16):
                        p=bl(d); loss=cr(p["mortality"],d.y_mortality.to(device))
                    if not(torch.isnan(loss)or torch.isinf(loss)): bls.append(loss)
                except: continue
            if bls:
                tot=torch.stack(bls).mean(); scaler.scale(tot).backward()
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(bl.parameters(),cfg.max_grad_norm)
                scaler.step(opt); scaler.update(); opt.zero_grad()
    bl.eval(); ym,pm=[],[]
    with torch.no_grad():
        for i in range(len(test_ds)):
            try:
                d=test_ds[i]; p=bl(d); ym.append(d.y_mortality.item()); pm.append(torch.sigmoid(p["mortality"]).item())
            except: continue
    ya,pa=np.array(ym),np.array(pm)
    a,lo,hi=bci(ya,pa,roc_auc_score)
    baselines.append({"name":f"GATv2_s{sd}","auroc":a,"ci":[lo,hi]})
    gat_probs.append(pa)
    np.save(f"{cfg.output_dir}/raw/p_gatv2_s{sd}.npy",pa)
    log.info(f"    AUROC={a:.4f}"); del bl,opt; flush()

if len(gat_probs)>=2:
    mn=min(len(p) for p in gat_probs)
    pa=np.stack([p[:mn] for p in gat_probs])
    yr=np.array([test_ds[i].y_mortality.item() for i in range(mn)])
    pm_=pa.mean(0); ps_=pa.std(0)
    a,lo,hi=bci(yr,pm_,roc_auc_score)
    baselines.append({"name":"DeepEnsemble","auroc":a,"ci":[lo,hi],"brier":float(brier_score_loss(yr,pm_))})
    np.save(f"{cfg.output_dir}/raw/p_ens.npy",pm_); np.save(f"{cfg.output_dir}/raw/p_ens_std.npy",ps_)
    log.info(f"  DeepEnsemble: AUROC={a:.4f}")
save_json(baselines, f"{cfg.output_dir}/results/baselines.json")

print("\n" + "="*70 + "\nSTEP 4: NoReasoning ABLATION\n" + "="*70)

class NoReas(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.enc = HeteroEncoder(FEAT_DIMS,c.hidden_dim,c.num_heads,c.num_gat_layers,c.graph_embed_dim,c.dropout,c.mc_dropout)
        self.nf = NoteFusion(c.graph_embed_dim,c.note_embed_dim)
        ed=c.graph_embed_dim; hd=c.hidden_dim
        self.bypass = nn.Sequential(nn.Linear(ed,hd),nn.GELU(),nn.LayerNorm(hd),nn.Linear(hd,hd),nn.GELU(),
                                     nn.Dropout(c.dropout),nn.Linear(hd,ed//2),nn.GELU(),nn.Linear(ed//2,ed),nn.LayerNorm(ed))
        self.heads = Heads(c.graph_embed_dim,c.mc_dropout)
        self.sg = nn.Parameter(torch.tensor(-10.0)); self.lt = nn.Parameter(torch.tensor(0.0))
        log.info(f"  NoReas params: {sum(p.numel() for p in self.parameters()):,}")
    def _unpack(self, data, zt=False):
        xd,td,eid={},{},{}
        for nt in data.node_types:
            if nt=="note": continue
            xd[nt]=data[nt].x.to(device)
            if hasattr(data[nt],"t") and not zt: td[nt]=data[nt].t.to(device)
        for et in data.edge_types:
            s,r,d=et
            if s=="note" or d=="note" or "xmodal" in r: continue
            eid[et]=data[et].edge_index.to(device)
        return xd,td,eid
    def _notes(self,data):
        is_text = hasattr(data['note'], 'text')
        nx = data['note'].text if is_text else data['note'].x.to(device)
        return nx, is_text
    def forward(self, data, tau=1.0, **kw):
        xd,td,eid=self._unpack(data)
        _,hG=self.enc(xd,eid,td)
        nx, is_text = self._notes(data)
        hf=self.nf(hG,nx,is_text=is_text)
        ho=self.bypass(hf)
        z=torch.tensor(0.0,device=device)
        return self.heads(ho),z,z,None,None,None
    @torch.no_grad()
    def mc_factored(self,data,Kd=5,Kg=5,tau=1.0,**kw):
        T=self.lt.exp().clamp(0.5,5.0).item(); mp=[]
        for _ in range(Kd*Kg):
            p,_,_,_,_,_=self.forward(data,tau)
            mp.append(torch.sigmoid(torch.tensor(p["mortality"].item()/T)).item())
        mp=np.array(mp); m=float(mp.mean())
        def H(p): p=np.clip(p,1e-8,1-1e-8); return -(p*np.log(p)+(1-p)*np.log(1-p))
        ut=H(m); ua=float(np.mean([H(x) for x in mp]))
        nn_=sum(data[nt].x.shape[0] for nt in data.node_types if nt!="note")
        return {"mortality_mean":m,"u_total":ut,"u_aleatoric":ua,"u_epistemic":max(0,ut-ua),"u_structural":0,
                "mort_std":float(mp.std()),"q_var":0,"edge_var":0,"n_nodes":nn_,"skip_gate":0}

ablations = {}
ck_nr=f"{cfg.output_dir}/checkpoints/noreas.pt"
m_nr,_,_ = train_seed(cfg.seeds[0],NoReas,nep=cfg.ablation_epochs,label="NoReas",ckpt=ck_nr)
df_nr = eval_models([m_nr],test_ds,desc="NoReas")
ag_nr = avg_stay(df_nr)
a,lo,hi = bci(ag_nr.y_mort.values,ag_nr.p_mort.values,roc_auc_score)
ablations["NoReasoning_trained"]={"auroc":a,"ci":[lo,hi]}
np.save(f"{cfg.output_dir}/raw/p_noreas.npy",ag_nr.p_mort.values); log.info(f"  NoReasoning: AUROC={a:.4f}")
del m_nr; flush()

print("\n" + "="*70 + "\nSTEP 5: EVALUATION\n" + "="*70)

test_df = eval_models(tegot_ms, test_ds, desc="Test")
test_df.to_csv(f"{cfg.output_dir}/results/test_all.csv",index=False)

per_seed = {}
for si,sd in enumerate(cfg.seeds):
    ds=test_df[test_df.seed==si]
    per_seed[f"s{sd}"] = metrics(ds, CONST_PRED)
save_json(per_seed, f"{cfg.output_dir}/results/per_seed.json")
seed_aurocs = [per_seed[f"s{sd}"].get("auroc",{}).get("mean") for sd in cfg.seeds if per_seed[f"s{sd}"].get("auroc")]
log.info(f"Per-seed AUROC: {[f'{a:.4f}' for a in seed_aurocs]}")

test_ens = avg_stay(test_df)
test_ens.to_csv(f"{cfg.output_dir}/results/test_ens.csv",index=False)
val_df = eval_models(tegot_ms, val_model, desc="Val")
val_ens = avg_stay(val_df)

tm, vm = metrics(test_ens, CONST_PRED), metrics(val_ens, CONST_PRED)
tm["seeds_mean"]=float(np.mean(seed_aurocs)); tm["seeds_std"]=float(np.std(seed_aurocs))
tm["n_params"]=sum(p.numel() for p in tegot_ms[0].parameters())
tm["skip_gate"]=float(torch.sigmoid(tegot_ms[0].sg).item())
va=vm.get("auroc",{}).get("mean"); ta=tm.get("auroc",{}).get("mean")
if va and ta: tm["vt_gap"]=va-ta; log.info(f"Val-Test gap: {va-ta:+.4f}")

save_json(tm, f"{cfg.output_dir}/results/test_metrics.json"); save_json(vm, f"{cfg.output_dir}/results/val_metrics.json")
np.save(f"{cfg.output_dir}/raw/p_tegot.npy",test_ens.p_mort.values); np.save(f"{cfg.output_dir}/raw/y_test.npy",test_ens.y_mort.values)
log.info(f"TEGoT AUROC: {tm['auroc']['mean']:.4f} [{tm['auroc']['ci_lo']:.4f},{tm['auroc']['ci_hi']:.4f}]")

print("\n" + "="*70 + "\nSTEP 6: INFERENCE ABLATIONS\n" + "="*70)

def quick_abl(desc, mn=False, zt=False):
    rows=[]
    for mi,m in enumerate(tegot_ms):
        m.eval()
        for j in range(len(test_ds)):
            try:
                d=test_ds[j]; mc=m.mc_factored(d,Kd=3,Kg=3,tau=cfg.mc_eval_tau,mn=mn,zt=zt)
                rows.append({"stay_id":test_ds.stay_ids[j],"y_mort":d.y_mortality.item(),"p_mort":mc["mortality_mean"],"seed":mi})
            except: continue
    df=pd.DataFrame(rows); ag=df.groupby("stay_id").agg({"y_mort":"first","p_mort":"mean"}).reset_index()
    a,lo,hi=bci(ag.y_mort.values,ag.p_mort.values,roc_auc_score)
    return {"auroc":a,"ci":[lo,hi]},ag.p_mort.values

r,pv=quick_abl("NoNotes",mn=True); ablations["NoNotes"]=r
np.save(f"{cfg.output_dir}/raw/p_nonotes.npy",pv); log.info(f"  NoNotes: {r['auroc']:.4f}")
r,pv=quick_abl("NoTemp",zt=True); ablations["NoTemporal"]=r
np.save(f"{cfg.output_dir}/raw/p_notemp.npy",pv); log.info(f"  NoTemporal: {r['auroc']:.4f}")
save_json(ablations, f"{cfg.output_dir}/results/ablations.json")

print("\n" + "="*70 + "\nSTEP 7: CLINICAL + STATS\n" + "="*70)

yt, pt = test_ens.y_mort.values, test_ens.p_mort.values
thr = youden(val_ens.y_mort.values,val_ens.p_mort.values)
save_json({"tuned":clin(yt,pt,thr),"fixed_0.5":clin(yt,pt,0.5)}, f"{cfg.output_dir}/results/clinical.json")

dca_rows = []; prev=yt.mean()
for t in np.arange(0.01,0.5,0.02):
    pr=(pt>=t).astype(int)
    tp=((pr==1)&(yt==1)).sum(); fp=((pr==1)&(yt==0)).sum(); n=len(yt)
    nb=(tp/n)-(fp/n)*(t/(1-t)); nba=prev-(1-prev)*(t/(1-t))
    dca_rows.append({"t":float(t),"nb":float(nb),"nba":float(nba)})
save_json(dca_rows, f"{cfg.output_dir}/results/dca.json")

st_tests = {}; pt_ = test_ens.p_mort.values
for name,fname in [("LR","p_lr.npy"),("MLP","p_mlp.npy"),("DeepEnsemble","p_ens.npy"),("NoNotes","p_nonotes.npy"),("NoTemporal","p_notemp.npy"),("NoReasoning","p_noreas.npy")]:
    fp=f"{cfg.output_dir}/raw/{fname}"
    if not os.path.exists(fp): continue
    pb=np.load(fp); nc=min(len(pt_),len(pb))
    d,lo,hi,pv=bpaired(yt[:nc],pt_[:nc],pb[:nc],roc_auc_score)
    if d is not None:
        st_tests[f"vs_{name}"]={"d":d,"ci":[lo,hi],"p":pv,"sig":pv<0.05}
save_json(st_tests, f"{cfg.output_dir}/results/stat_tests.json")

pbin=(pt>=thr).astype(int); is_wrong=(pbin!=yt.astype(int)).astype(int); uu={}
for uc in ["u_total","u_epistemic","u_structural","u_aleatoric","mort_std"]:
    if uc in test_ens.columns and np.std(test_ens[uc])>0 and len(np.unique(is_wrong))>=2:
        uu[f"{uc}_errAUROC"]=float(roc_auc_score(is_wrong,test_ens[uc].values))
if "u_structural" in test_ens.columns and "n_nodes" in test_ens.columns:
    us=test_ens.u_structural.values; nn_=test_ens.n_nodes.values
    if np.std(nn_)>0 and np.std(us)>0:
        from numpy.polynomial import polynomial as P
        coef=P.polyfit(nn_,us,1); us_r=us-P.polyval(nn_,coef)
        if np.std(us_r)>0 and len(np.unique(is_wrong))>=2: uu["u_struct_resid_errAUROC"]=float(roc_auc_score(is_wrong,us_r))
save_json(uu, f"{cfg.output_dir}/results/uncertainty.json")

print("\n" + "="*70 + "\nSTEP 8: PLOTS\n" + "="*70)
pdir=f"{cfg.output_dir}/plots"


log.info("🎉 DONE. PIPELINE EXECUTED SUCCESSFULLY.")

09:00:15 | Downloading and initializing BioMistral-7B with MIMIC-IV LoRA...
09:00:42 | Fetching 18 files: 100%|██████████| 18/18 [00:26<00:00,  1.42MB/s]
09:01:22 | ✓ BioMistral initialized successfully.

STEP 1: TABULAR BASELINES (LR + MLP)
09:01:25 | Bundle meta: n_dx=26, precomputed_note_dim=384
09:01:28 | Mortality rate: 0.150, pos_weight: 3.00
09:01:35 |   Train: 8000 graphs
09:01:39 |   Val: 1500 graphs
09:01:43 |   Test: 1500 graphs
09:01:44 |   Val-Calib: 300
09:01:44 |   Val-Model: 1200
09:02:10 | Tabular: X_tr=(8000, 470), X_te=(1500, 470)
09:02:45 |   LR: AUROC=0.8430 [0.8180,0.8670]
09:03:12 |   MLP: AUROC=0.8710

STEP 2: TEGoT v16 × SEEDS
09:03:18 |   TEGoT params: 975,289
09:18:33 |   [TEGoT s42] ep1/15 | tr=0.9241 val=0.8813 | β=0.0000 kl=17.330 gn=28.4 sg=0.121 | 915s | ★
09:33:40 |   [TEGoT s42] ep2/15 | tr=0.8714 val=0.8092 | β=0.0040 kl=8.720 gn=19.2 sg=0.138 | 907s | ★
09:48:52 |   [TEGoT s42] ep3/15 | tr=0.8531 val=0.7644 | β=0.0080 kl=6.940 gn=14.4 sg=0.184 | 912s